# 1.6 章节实践：单变量性能优化

## 本节学习目标

本实践要求综合运用本章知识完成可复现的工程任务。请保留命令、参数、正确性结果和分析结论。

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v msprof
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 必要背景与实验材料

本实践基于本章 `src/` 中的课程工程副本。开始前应完成前面各小节，并能解释工程的关键源码、构建入口、正确性门槛和计时字段。

## 实践任务

1. 选择 partition（rows|nnz）、rank 数或向量操作配置中的一个变量
2. 记录 baseline 的正确性、total、阶段时间和 collective 次数
3. 只修改该变量并重新测量
4. 比较前后结果、检查正确性并解释改善或退化

## 核心知识与关键源码解析

本章实践只调整 Notebook 中的运行参数和实验配置，不修改 `src/`。请保持输入与正确性阈值一致，每次只改变一个变量，并说明它如何影响数据流和性能。

## 实验记录模板

执行下面的准备命令，然后在目标环境中完成任务。不要把参考答案中的结论当作实测结果。

In [ ]:
%%bash
set -euo pipefail
cd src/dis_gmres
DIS_GMRES_REQUIRE_REAL=1 bash scripts/build.sh
# Baseline：固定矩阵、rank、线程预算和重复次数。
bash scripts/run.sh --npus 2 --matrix U1 --warmup 0 --repeat 10 --orthogonalization mgs --partition rows
# 单变量复测：只把 partition 从 rows 改为 nnz。
bash scripts/run.sh --npus 2 --matrix U1 --warmup 0 --repeat 10 --orthogonalization mgs --partition nnz

# CGS 能力探测：成功时核对真实 Device 与正确性字段；不支持时核对明确拒绝信息。
set +e
bash scripts/run.sh --no-baseline --npus 2 --matrix U1 --warmup 0 --repeat 3 --orthogonalization cgs > results/cgs_probe.log 2>&1
cgs_status=$?
set -e
if [[ "$cgs_status" -eq 0 ]]; then
  grep -Fq 'compute backend = Ascend C RTC Device GMRES' results/cgs_probe.log
  grep -Fq 'solution relative error = ' results/cgs_probe.log
  grep -Eq '^RESULT_RESIDUAL=[0-9.eE+-]+$' results/cgs_probe.log
  echo 'CGS_CAPABILITY=SUPPORTED correctness=PASS backend=Ascend C RTC Device GMRES'
elif grep -Fq 'Device CGS multi-dot kernel is not enabled' results/cgs_probe.log || \
     grep -Fq 'Device CGS multi-dot kernel is not enabled' results/rank_*.log 2>/dev/null; then
  echo "CGS_CAPABILITY=UNSUPPORTED exit_code=$cgs_status reason=device_multi_dot_not_enabled"
else
  cat results/cgs_probe.log >&2
  for log in results/rank_*.log; do [[ -f "$log" ]] && { echo "===== $log =====" >&2; cat "$log" >&2; }; done
  echo "CGS_CAPABILITY=ERROR exit_code=$cgs_status reason=unexpected_failure" >&2
  exit 1
fi


## 评价标准

必须形成 baseline 到修改后的闭环；若性能退化，同样可得到有效结论，但需用 profile 证据解释。

## 查看参考答案

参考答案给出方法和判断依据，不提供虚构的固定性能数字。

## 预期现象与结果分析

正确性门槛应首先通过；性能结果随硬件、软件栈和系统负载变化。若修改后没有加速或出现退化，也应依据阶段计时、通信次数或资源竞争给出解释。

## 实践小结

完成报告时，应明确实验环境、唯一修改变量、正确性门槛、计时口径和观察到的限制。

## 工程实践提交物与完成标准

章测必须基于 `src/dis_gmres/`，不得只回答概念题。操作链：构建测试 → baseline → 核对 residual/iterations → 读 profiling → 找瓶颈 → 选择一个变量 → 只改该变量重跑 → 比较结论。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“配置、Total、SpMV、Dot、AXPY、Norm、HCCL、Transfer、Residual、Iterations、Speedup”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 四类考核

以下四题中，客观题答案唯一，凭 `src/dis_gmres/` 源码与本 Notebook 的能力探测逻辑即可判定；简单/中等/困难题基于本章实验，要求用命令、输出、CSV 或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：当前 Device GMRES 在每次 SpMV 前，全局向量是如何形成的（　）

A. 每个 rank 把本地向量经 HCCL AllGather 汇总成全局向量

B. 每个 rank 把本地向量放入全局 Device buffer，再经 HCCL AllReduce 形成全局向量

C. Host 把所有 rank 的本地向量拼接后回传 Device

D. 各 rank 各自维护完整全局向量，不做任何通信

（2）判断（对/错）：章测对 `--orthogonalization cgs` 做能力探测：运行成功时必须核对真实 Device backend 与正确性字段；不支持时必须匹配明确拒绝信息；两种情况都不允许静默回退 CPU。（　）

（3）单选：`--warmup 0` 的语义是（　）

A. 关闭每次求解重建 solver 状态的路径，只测稳态

B. 矩阵读取/分区与 communicator 初始化在 repeat 外只做一次；每次 solve 重建 solver 内的 RTC/Device 状态；warmup 只是丢弃额外 solver 调用，不代表 warm cache 稳态指标，也不等于完整进程冷启动

C. 跳过正确性校验

D. 只运行 0 次 solve

### 2. 简单题

给出一次 solve 的完整命令（`--warmup 0`）与日志证据：backend、compute backend、device、residual、iterations、total time；说明矩阵读取/分区与 communicator 初始化位于 repeat 外，每次 solve 重建 solver 内的 RTC/Device 状态，`--warmup` 只是丢弃额外 solver 调用、不代表 warm cache 稳态指标。

### 3. 中等题

运行 `msprof` 导出一份 Summary/Timeline CSV，用 Python 标准库按名称列累计耗时（ns 转 us），输出 Top 10 阶段、总时长与 share%，结合 MGS 数据流（局部向量 → 全局 Device buffer → HCCL AllReduce → Device SpMV）指出第一瓶颈；必须引用本次导出 CSV 的列名与数值。

### 4. 困难题

做一次 2 rank 正确性+性能实验：给出每 rank 的 residual/error、总时间与通信占比，与同参数 1 rank 对比并解释扩展性；再选择一个单变量（如 orthogonalization 或 partition）完成假设-验证闭环，前后对比必须来自实测 CSV/日志。
